# Job-board applications analysis (July–September 2026)

This notebook cleans the supplied synthetic dataset and answers the three assignment questions. Run all cells from top to bottom. Keep `job-board-applications.csv` in the same folder as this notebook.

**Cleaning decisions**
- Remove duplicate `application_id` records (the six repeated IDs are exact duplicate rows).
- Parse `applied_on` as a date.
- Standardise text fields to lowercase and map `on-site` to `on_site`.
- Label missing `source` as `unknown`; do not infer the missing source.
- Preserve missing `days_to_first_reply` as missing because the brief defines blanks as no reply yet.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("job-board-applications.csv")
print("Raw shape:", df.shape)
display(df.head())


In [ ]:
# Data-quality checks before cleaning
print("Missing values:\n", df.isna().sum())
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate application IDs:", df["application_id"].duplicated().sum())
print("Application date range:", df["applied_on"].min(), "to", df["applied_on"].max())
print("Steps range:", df["steps_done"].min(), "to", df["steps_done"].max())
print("Stipend range:", df["stipend_inr_month"].min(), "to", df["stipend_inr_month"].max())


In [ ]:
# Clean the data
clean = df.drop_duplicates(subset="application_id", keep="first").copy()
clean["applied_on"] = pd.to_datetime(clean["applied_on"], errors="coerce")

for col in ["role_category", "work_mode", "source", "shortlisted", "interviewed", "hired"]:
    clean[col] = clean[col].astype("string").str.strip().str.lower()

clean["work_mode"] = clean["work_mode"].replace({"on-site": "on_site", "onsite": "on_site"})
clean["source"] = clean["source"].fillna("unknown")
clean["days_to_first_reply"] = pd.to_numeric(clean["days_to_first_reply"], errors="coerce")

print("Cleaned shape:", clean.shape)
print("Unique application IDs:", clean["application_id"].nunique())
print("\nWork mode values:\n", clean["work_mode"].value_counts())
print("\nSource values including unknown:\n", clean["source"].value_counts())
print("\nRemaining missing values:\n", clean.isna().sum())


## Question 1 – Hires per 100 applications by source

In [ ]:
q1 = clean.groupby("source").agg(
    applications=("application_id", "count"),
    hires=("hired", lambda s: (s == "yes").sum())
)
q1["hires_per_100"] = q1["hires"] / q1["applications"] * 100
q1 = q1.sort_values("hires_per_100", ascending=False)
display(q1.round(2))

q1_role = clean.groupby(["role_category", "source"]).agg(
    applications=("application_id", "count"),
    hires=("hired", lambda s: (s == "yes").sum())
)
q1_role["hires_per_100"] = q1_role["hires"] / q1_role["applications"] * 100
display(q1_role.reset_index().sort_values(
    ["role_category", "hires_per_100"], ascending=[True, False]
).round(2))


## Question 2 – Shortlisting by completion of all four steps

In [ ]:
clean["completed_all_4"] = clean["steps_done"].eq(4)
q2 = clean.groupby("completed_all_4").agg(
    applications=("application_id", "count"),
    shortlisted=("shortlisted", lambda s: (s == "yes").sum())
)
q2["shortlist_rate_pct"] = q2["shortlisted"] / q2["applications"] * 100
display(q2.round(2))

rate4 = q2.loc[True, "shortlist_rate_pct"]
rate_lt4 = q2.loc[False, "shortlist_rate_pct"]
print(f"Difference: {rate4 - rate_lt4:.2f} percentage points")
print(f"Rate ratio: {rate4 / rate_lt4:.2f}x")


In [ ]:
labels = ["Fewer than 4 steps", "All 4 steps"]
values = [rate_lt4, rate4]
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels, values)
ax.set_title("Shortlisting rate by application-step completion")
ax.set_ylabel("Applicants shortlisted (%)")
ax.set_ylim(0, max(values) * 1.25)
ax.grid(axis="y", alpha=0.25)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f"{val:.1f}%", ha="center", va="bottom")
plt.tight_layout()
plt.show()


## Question 3 – Reply times and no-reply share for paid vs unpaid roles

In [ ]:
clean["pay_status"] = np.where(clean["stipend_inr_month"] > 0, "paid", "unpaid")
q3 = clean.groupby("pay_status").agg(
    applications=("application_id", "count"),
    no_reply=("days_to_first_reply", lambda s: s.isna().sum()),
    replied=("days_to_first_reply", lambda s: s.notna().sum()),
    mean_days_replied=("days_to_first_reply", "mean"),
    median_days_replied=("days_to_first_reply", "median")
)
q3["no_reply_share_pct"] = q3["no_reply"] / q3["applications"] * 100
display(q3.round(2))


## Interpretation and limitations

- The source with the highest overall hire rate is not the leader in every role category.
- Four-step completion is strongly associated with shortlisting, but this observational dataset cannot establish causality.
- Paid roles show shorter reply times and a lower no-reply share in this dataset. Do not interpret this as proof that paying a stipend causes faster replies.
- The dataset does not provide a common observation cut-off. Applications submitted later may have had less time to receive a reply.
- Small source-by-role groups can produce unstable hire rates; always show denominators alongside rates.


In [ ]:
# Export the cleaned dataset for reuse
clean.to_csv("cleaned_job_board_applications.csv", index=False)
print("Saved cleaned_job_board_applications.csv")
